# Tour: `baselode.drill` end-to-end

Walks through the full Python library against GSWA open geochemistry test data — loaders, mapping, strip logs, desurveying, compositing, interval QC, comprehensive validation, the `DrillholeSet` composition root, and OMF export.

**Sections covered:**
1. Setup & data loading — collars, surveys, assays, structures
2. Collar map
3. Strip logs (numeric / categorical / comments / tadpole)
4. Desurveying (minimum curvature / tangential / balanced tangential)
5. Attaching 3D positions
6. Compositing
7. Structural geometry helpers
8. Interval algebra — gap / overlap / split / merge
9. Comprehensive validation
10. `DrillholeSet` — the composition root
11. OMF export


## Install baselode

Into your venv or conda-env:

1. From pypi

`pip install baselode`

2. From a local build

`cd python/src`
`python -m build`
`pip install baselode<ver>.whl`

3. Locally in dev mode

`python -m pip install -e python/src`

In [1]:
# Uncomment to install locally in dev mode
! pwd
! python -m pip install ../python

/Users/tam/Code/darkmine/darkmine-oss/baselode/notebooks


Processing /Users/tam/Code/darkmine/darkmine-oss/baselode/python


  Installing build dependencies ... -

 \ done


  Getting requirements to build wheel ... -

...

 \

 done


 \

 done
  Created wheel for baselode: filename=baselode-0.1.26.post1-py3-none-any.whl size=115561 sha256=1803244318955d177b729dbec25918f20b52c460d4734d715abf9dbe7e7852fb
  Stored in directory: /private/var/folders/j4/w99gsg556lz4m05d6rbr21580000gn/T/pip-ephem-wheel-cache-s0of0c42/wheels/81/db/ad/7ca807253354d55bd74c41791f8b8564071ba6c293336b1286
Successfully built baselode


  Attempting uninstall: baselode
    Found existing installation: baselode 0.1.26.post1
    Uninstalling baselode-0.1.26.post1:
      Successfully uninstalled baselode-0.1.26.post1

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pathlib
import folium
import pandas as pd
import plotly.graph_objects as go

from baselode.drill.data import (
    load_collars, load_surveys, load_assays, load_structures,
)
from baselode.drill.desurvey import (
    minimum_curvature_desurvey,
    tangential_desurvey,
    balanced_tangential_desurvey,
    attach_assay_positions,
    build_traces,
)
from baselode.drill.composite import composite_intervals, resample_trace
from baselode.drill.structural import (
    normalize_dip_azimuth,
    compute_strike,
    compute_plane_normal,
    attach_structure_positions,
)
from baselode.drill.view import (
    plot_drillhole_trace,
    plot_strip_log,
    plot_point_log,
    plot_comments_log,
    plot_tadpole_log,
    compute_interval_points,
)
from baselode.map import create_leaflet_map, map_collar_points, map_collars
from baselode.datamodel import HOLE_ID, COMMENTS, DEPTH


## Test Data — GSWA Open Geochemistry

Four CSV files from the Geological Survey of Western Australia open-file geochemistry dataset:

| File | Description |
|------|-------------|
| `gswa_sample_collars.csv` | Hole locations (lat/lon, elevation, 4 685 holes) |
| `gswa_sample_survey.csv` | Downhole survey measurements (dip/azimuth) |
| `gswa_sample_assays.csv` | Multi-element geochemistry (60+ elements, PPM) |
| `gswa_sample_structure.csv` | Structural point measurements (dip, azimuth) |

> See `test/data/gswa/README.md` and `DATA_LICENSE.md` for full attribution.

In [3]:
repo_root = pathlib.Path("..").resolve()
TEST_DIR = repo_root / "test" / "data" / "gswa"

COLLAR_CSV    = TEST_DIR / "gswa_sample_collars.csv"
SURVEY_CSV    = TEST_DIR / "gswa_sample_survey.csv"
ASSAY_CSV     = TEST_DIR / "gswa_sample_assays.csv"
STRUCTURE_CSV = TEST_DIR / "gswa_sample_structure.csv"

print("Test data directory:", TEST_DIR)
for p in [COLLAR_CSV, SURVEY_CSV, ASSAY_CSV, STRUCTURE_CSV]:
    print(f"  {'OK' if p.exists() else '!! MISSING':10s} {p.name}")

Test data directory: /Users/tam/Code/darkmine/darkmine-oss/baselode/test/data/gswa
  OK         gswa_sample_collars.csv
  OK         gswa_sample_survey.csv
  OK         gswa_sample_assays.csv
  OK         gswa_sample_structure.csv


### Collars

In [4]:
collar_gdf = load_collars(COLLAR_CSV)
print(f"Loaded {len(collar_gdf)} collars  |  CRS: {collar_gdf.crs}")
collar_gdf.head()

Loaded 4685 collars  |  CRS: EPSG:4326


,id,hole_id,anumber,datasource_hole_id,project_id,longitude,latitude,istransformed,modifieddate,modifiedby,...,maxdepth,collarid,elevation,elevation_uom,height_datum,height_vert_accuracy,height_horiz_accuracy,height_capture_method,last_updated,geometry
0,3131885,100289FORRESTANIA PROJECTFSRC044,100289,FSRC044,4072,119.633823,-32.363756,None,None,None,...,198.0,3131885,379.0,Metres,EGM96,99% of points are within a height difference o...,7.2m (90% of Australia),Drillhole collar location intersected with GA ...,2016-04-12,POINT (119.63382 -32.36376)
1,3131886,100289FORRESTANIA PROJECTFSRC045,100289,FSRC045,4072,119.63797,-32.351747,None,None,None,...,198.0,3131886,378.0,Metres,EGM96,99% of points are within a height difference o...,7.2m (90% of Australia),Drillhole collar location intersected with GA ...,2016-04-12,POINT (119.63797 -32.35175)
2,3131887,100289FORRESTANIA PROJECTFSRC046,100289,FSRC046,4072,119.629281,-32.351959,None,None,None,...,250.0,3131887,375.0,Metres,EGM96,99% of points are within a height difference o...,7.2m (90% of Australia),Drillhole collar location intersected with GA ...,2016-04-12,POINT (119.62928 -32.35196)
3,3131888,100289FORRESTANIA PROJECTFSRC047,100289,FSRC047,4072,119.623721,-32.359436,None,None,None,...,198.0,3131888,374.0,Metres,EGM96,99% of points are within a height difference o...,7.2m (90% of Australia),Drillhole collar location intersected with GA ...,2016-04-12,POINT (119.62372 -32.35944)
4,3131889,100289FORRESTANIA PROJECTFSRC048,100289,FSRC048,4072,119.629231,-32.35198,None,None,None,...,210.0,3131889,375.0,Metres,EGM96,99% of points are within a height difference o...,7.2m (90% of Australia),Drillhole collar location intersected with GA ...,2016-04-12,POINT (119.62923 -32.35198)


In [5]:
# The full hole_id is a concatenated key from the source database.
# Use datasource_hole_id to find the full hole_id from a short company name.
print(collar_gdf[["hole_id", "datasource_hole_id"]].drop_duplicates().head(10).to_string(index=False))

# Example: look up the full hole_id for company short name 'NMD140'
matches = collar_gdf[collar_gdf["datasource_hole_id"] == "NMD140"]
print("\nNMD140 full hole_id:", matches["hole_id"].tolist())


                           hole_id datasource_hole_id
  100289FORRESTANIA PROJECTFSRC044            FSRC044
  100289FORRESTANIA PROJECTFSRC045            FSRC045
  100289FORRESTANIA PROJECTFSRC046            FSRC046
  100289FORRESTANIA PROJECTFSRC047            FSRC047
  100289FORRESTANIA PROJECTFSRC048            FSRC048
  100289FORRESTANIA PROJECTFSRC059            FSRC059
101441Mt Gibb JV ForrestaniaCCD001             CCD001
            101533ForrestaniaBD048              BD048
            101533ForrestaniaBD049              BD049
            101533ForrestaniaBD050              BD050

NMD140 full hole_id: ['101533ForrestaniaNMD140', '97253ForrestaniaNMD140']


### Survey

In [6]:
survey_df = load_surveys(SURVEY_CSV)
print(f"Loaded {len(survey_df)} survey rows  |  {survey_df[HOLE_ID].nunique()} holes")
survey_df.head()

Loaded 158547 survey rows  |  3996 holes


,id,collarid,depth,units,accuracy,loaddate,loadby,modifieddate,modifiedby,mrtfileid,dip,azimuth,hole_id,datasource_hole_id
0,9130283,3210874,0.0,NaN,NaN,2014-04-16 13:48:17.480,INTERNAL\MIGISJT,2024-06-22 10:08:29.490,INTERNAL\ZZDA_MICSBJC,92050,-59.900002,93.199997,101441Mt Gibb JV ForrestaniaCCD001,CCD001
1,9130284,3210874,5.0,NaN,NaN,2014-04-16 13:48:17.480,INTERNAL\MIGISJT,2024-06-22 10:08:29.490,INTERNAL\ZZDA_MICSBJC,92050,-59.900002,93.199997,101441Mt Gibb JV ForrestaniaCCD001,CCD001
2,9130285,3210874,10.0,NaN,NaN,2014-04-16 13:48:17.480,INTERNAL\MIGISJT,2024-06-22 10:08:29.490,INTERNAL\ZZDA_MICSBJC,92050,-59.799999,93.000000,101441Mt Gibb JV ForrestaniaCCD001,CCD001
3,9130286,3210874,15.0,NaN,NaN,2014-04-16 13:48:17.480,INTERNAL\MIGISJT,2024-06-22 10:08:29.490,INTERNAL\ZZDA_MICSBJC,92050,-60.000000,92.900002,101441Mt Gibb JV ForrestaniaCCD001,CCD001
4,9130287,3210874,20.0,NaN,NaN,2014-04-16 13:48:17.480,INTERNAL\MIGISJT,2024-06-22 10:08:29.490,INTERNAL\ZZDA_MICSBJC,92050,-60.000000,92.699997,101441Mt Gibb JV ForrestaniaCCD001,CCD001


### Assays

In [7]:
assay_df = load_assays(ASSAY_CSV)
element_cols = [c for c in assay_df.columns if c.endswith("_ppm")]
print(f"Loaded {len(assay_df)} assay rows  |  {assay_df[HOLE_ID].nunique()} holes  |  {len(element_cols)} elements")
assay_df.head()

Loaded 69420 assay rows  |  3576 holes  |  53 elements


/Users/tam/Code/darkmine/darkmine-oss/baselode/.venv/lib/python3.12/site-packages/baselode/drill/data.py:222: DtypeWarning: Columns (0: simplifiedMethod, 1: simplifiedDigest) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(source, **kwargs)


,id,latitude,longitude,anumber,collarid,companyholeid_x,from,to,ag_ppm,al_ppm,...,dip,azimuth,holetype,shape,elevation,simplifiedmethod,simplifieddigest,hole_id,companyholeid_y,mid
0,40689304,-32.363756,119.633823,100289,3131885,FSRC044,0.0,4.0,NaN,99000.0,...,-59.501465,331.984070,RC,POINT (119.6338175425 -32.3637482749 -1.723284...,380.0,ICP,4_ACID,100289FORRESTANIA PROJECTFSRC044,FSRC044,2.0
1,40689305,-32.363756,119.633823,100289,3131885,FSRC044,4.0,8.0,NaN,125000.0,...,-59.501343,331.985382,RC,POINT (119.6338074576 -32.3637320936 -5.169846...,380.0,ICP,4_ACID,100289FORRESTANIA PROJECTFSRC044,FSRC044,6.0
2,40689306,-32.363756,119.633823,100289,3131885,FSRC044,8.0,12.0,NaN,121000.0,...,-59.501221,331.986725,RC,POINT (119.6337973735 -32.3637159118 -8.616399...,380.0,ICP,4_ACID,100289FORRESTANIA PROJECTFSRC044,FSRC044,10.0
3,40689307,-32.363756,119.633823,100289,3131885,FSRC044,12.0,16.0,NaN,128000.0,...,-59.501099,331.988037,RC,POINT (119.6337872902 -32.3636997295 -12.06294...,380.0,ICP,4_ACID,100289FORRESTANIA PROJECTFSRC044,FSRC044,14.0
4,40689308,-32.363756,119.633823,100289,3131885,FSRC044,16.0,20.0,NaN,94600.0,...,-59.500977,331.989380,RC,POINT (119.6337772078 -32.3636835466 -15.50948...,380.0,ICP,4_ACID,100289FORRESTANIA PROJECTFSRC044,FSRC044,18.0


### Structures

In [8]:
struct_df = load_structures(STRUCTURE_CSV)
print(f"Loaded {len(struct_df)} structural points  |  {struct_df[HOLE_ID].nunique()} holes")
print(f"Dip range: {struct_df['dip'].min():.1f}° – {struct_df['dip'].max():.1f}°  |  "
      f"Azimuth range: {struct_df['azimuth'].min():.1f}° – {struct_df['azimuth'].max():.1f}°")
struct_df.head()

Loaded 161241 structural points  |  866 holes
Dip range: 0.0° – 90.0°  |  Azimuth range: 0.0° – 360.0°


,id,hole_id,depth,alpha,beta,dip,azimuth,defect,defect_width,description
155960,1761794,101533ForrestaniaBD053,71.139999,32.0,225.0,NaN,NaN,fr,NaN,NaN
155961,1761795,101533ForrestaniaBD053,71.809998,26.0,90.0,NaN,NaN,jt,NaN,NaN
155962,1761796,101533ForrestaniaBD053,76.320000,43.0,130.0,NaN,NaN,jt,NaN,NaN
155963,1761797,101533ForrestaniaBD053,76.559998,61.0,223.0,NaN,NaN,fr,NaN,NaN
155964,1761798,101533ForrestaniaBD053,91.330002,34.0,163.0,NaN,NaN,vc,NaN,Alteration-banding-veining


## Collar Map

Interactive map of all collar locations coloured by elevation. Requires an internet connection for the OpenStreetMap tile layer.

In [9]:
m = map_collars(collar_gdf)
print(f"Mapped {len(collar_gdf)} collars")
m

# In VS Code you may need to configure 'Manage Workspace Trust' and add the folder
# for this notebook to get the map to display. Then close/reopen the notebook.


Mapped 4685 collars

## Strip Logs

Each strip log type demonstrated with GSWA real data. A single representative hole is selected for each.

### Numeric — Assay (Au PPM)

In [10]:
# Hole has Au data and matching collar + survey records
# hole_id is the full concatenated key; datasource_hole_id holds the short company hole id 
NUMERIC_HOLE = "81604ForrestaniaWWRC015"
df_assay_hole = assay_df[assay_df[HOLE_ID] == NUMERIC_HOLE]
print(f"{NUMERIC_HOLE}: {len(df_assay_hole)} intervals, {df_assay_hole['au_ppm'].notna().sum()} Au values")

# here we use subplots, which can be zoomed in tandem. 
from plotly.subplots import make_subplots

fig1 = plot_drillhole_trace(df=df_assay_hole, value_col="au_ppm", chart_type="markers+line", intervals=False)
fig2 = plot_drillhole_trace(df=df_assay_hole, value_col="au_ppm", chart_type="markers+line", intervals=True)
fig3 = plot_drillhole_trace(df=df_assay_hole, value_col="au_ppm", chart_type="markers", intervals=False)
fig4 = plot_drillhole_trace(df=df_assay_hole, value_col="au_ppm", chart_type="line", intervals=False)
fig5 = plot_drillhole_trace(df=df_assay_hole, value_col="au_ppm", chart_type="bar", intervals=True)

combined = make_subplots(rows=1, cols=5, shared_yaxes=True,
subplot_titles=["markers+line, intervals=False", "markers+line, intervals=True", "markers, intervals=False", "line, intervals=False", "bar, intervals=True"])
for trace in fig1.data:
    combined.add_trace(trace, row=1, col=1)
for trace in fig2.data:
    combined.add_trace(trace, row=1, col=2)
for trace in fig3.data:
    combined.add_trace(trace, row=1, col=3)
for trace in fig4.data:
    combined.add_trace(trace, row=1, col=4)
for trace in fig5.data:
    combined.add_trace(trace, row=1, col=5)

combined.update_layout(width=1500, height=700, showlegend=False,
paper_bgcolor="white", plot_bgcolor="white")
combined.update_xaxes(title_text="au_ppm", gridcolor="#e8e8e8", linecolor="#d0d0d0")
combined.update_yaxes(autorange="reversed", gridcolor="#e8e8e8", linecolor="#d0d0d0")
combined.update_yaxes(title_text="Depth (m)", row=1, col=1)

combined.update_layout(title_text=f"baselode striplog numeric chart types. intervals show the full length of the sample")

combined.show()



81604ForrestaniaWWRC015: 24 intervals, 24 Au values


### Categorical — Structural Defect Types

`plot_strip_log` renders categorical intervals as coloured bands. Structural measurements are point data, so we construct a 0.1 m pseudo-interval (`to = depth + 0.1`) to display each measurement as a band.

In [11]:
# Pick the hole with the most labelled point structures
CAT_HOLE = struct_df[struct_df["defect"].notna()][HOLE_ID].value_counts().index[0]
df_cat = struct_df[struct_df[HOLE_ID] == CAT_HOLE].copy()

# bit of a hack to show how to do categroical graphs prior to true interval categroical data source
df_cat["from"] = df_cat[DEPTH]
df_cat["to"]   = df_cat[DEPTH] + 1.0

print(f"{CAT_HOLE}: {df_cat['defect'].notna().sum()} measurements with defect type")
print("Defect types:", sorted(df_cat["defect"].dropna().unique().tolist()))

fig = plot_strip_log(df=df_cat, from_col="from", to_col="to", label_col="defect")
fig.update_layout(width=300, height=700)
fig.show()

72183ForrestaniaCBD196: 1379 measurements with defect type
Defect types: ['bn', 'co', 'fn', 'fr', 'ft', 'jt', 'vn']


In [12]:
# plot_point_log — categorical point data with unique x-position, colour and marker per category
fig = plot_point_log(
    df=df_cat,
    depth_col=DEPTH,
    label_col="defect",
    marker_size=8,
)
fig.update_layout(height=700, width=300)
fig.show()


### Comments — Structural Descriptions

`plot_comments_log` renders depth intervals that carry text as labelled rectangles. Here we use the structural `description` column.

In [13]:
# Pick the hole with the most non-null descriptions
DESC_HOLE = struct_df[struct_df['description'].notna()][HOLE_ID].value_counts().index[0]
print(DESC_HOLE)
df_desc = struct_df[struct_df[HOLE_ID] == DESC_HOLE].copy()
df_desc = df_desc[df_desc['description'].notna()]

# in this dataset, comments are at a specific depth rather than applying to an interval, 
# so we make 0.1m intervals to mock it up as this code currently only does comments for intervals
# will do comments for point locations in future
df_desc["from"] = df_desc[DEPTH]
df_desc["to"]   = df_desc[DEPTH] + 0.1
print(f"{DESC_HOLE}: {len(df_desc)} measurements with descriptions")

fig = plot_comments_log(
    df=df_desc,
    from_col="from",
    to_col="to",
    comment_col='description',
)
fig.update_layout(height=700, width=300)
fig.show()

72183ForrestaniaSD045
72183ForrestaniaSD045: 549 measurements with descriptions


### Tadpole — Structural Dip & Azimuth

Each measurement is plotted as a circle at `(dip, depth)`. The tail length scales with dip magnitude and its direction encodes azimuth clockwise from North.

In [14]:

# Pick the hole with the most measurements that have both dip and azimuth values
tadpole_hole = (
    struct_df[struct_df["dip"].notna() & struct_df["azimuth"].notna()]
    [HOLE_ID].value_counts().index[0]
)
df_struct_hole = struct_df[struct_df[HOLE_ID] == tadpole_hole]

print(f"Hole: {tadpole_hole}  |  {len(df_struct_hole)} measurements with dip & azimuth")
print(df_struct_hole[["depth", "dip", "azimuth", "alpha", "beta"]].head())

fig = plot_tadpole_log(
    df=df_struct_hole,
    md_col="depth",
    dip_col="dip",
    az_col="azimuth",
    color_by="defect",
)

fig.show()


Hole: 72183ForrestaniaFFD167W5W1W1  |  1056 measurements with dip & azimuth
            depth   dip  azimuth  alpha   beta
50225  625.570007   NaN      NaN   85.0    NaN
50226  628.919983   NaN      NaN   78.0    NaN
50227  630.340027   NaN      NaN   45.0    NaN
50228  631.859985   NaN      NaN   79.0    NaN
50229  652.559998  29.5    359.8   45.0  220.0


## Desurveying

Full disclosure: these are not tested properly, but will be assessed against gslib and other implementations in future versions, and the best+fastest implementations will remain.

Converts survey measurements (depth, azimuth, dip) into 3D trace coordinates (easting, northing, elevation) from the collar origin. Three methods are available:

| Method | Description |
|--------|-------------|
| **Minimum curvature** | Industry standard — fits circular arc between stations |
| **Tangential** | Simpler — uses the starting orientation for the whole segment |
| **Balanced tangential** | Average of start/end orientations per segment |

First, we filter to the subset of holes that have both collar and survey data.

In [15]:
# Filter to holes that exist in both collars and surveys so desurvey has full inputs
collar_holes   = set(collar_gdf[HOLE_ID])
survey_holes   = set(survey_df[HOLE_ID])
shared_holes   = collar_holes & survey_holes

collars_matched = collar_gdf[collar_gdf[HOLE_ID].isin(shared_holes)].copy()
surveys_matched = survey_df[survey_df[HOLE_ID].isin(shared_holes)].copy()

print(f"Collars: {len(collar_gdf)} total  |  {len(collars_matched)} with matching survey")
print(f"Surveys: {len(survey_df)} total  |  {len(surveys_matched)} rows for matched holes")
print(f"Shared holes: {len(shared_holes)}")

Collars: 4685 total  |  3687 with matching survey
Surveys: 158547 total  |  154416 rows for matched holes
Shared holes: 3687


### Minimum Curvature (default)

In [16]:
traces_mc = minimum_curvature_desurvey(collars_matched, surveys_matched, step=5.0)
print(f"Minimum curvature: {len(traces_mc):,} trace points  |  {traces_mc[HOLE_ID].nunique()} holes")
traces_mc[traces_mc[HOLE_ID] == traces_mc[HOLE_ID].iloc[0]].head(10)

Minimum curvature: 170,165 trace points  |  3687 holes


,hole_id,md,easting,northing,elevation,azimuth,dip
0,101441Mt Gibb JV ForrestaniaCCD001,0.0,0.000000,0.000000,378.000000,93.199997,-59.900002
1,101441Mt Gibb JV ForrestaniaCCD001,5.0,2.503644,-0.139975,382.325757,93.199997,-59.900002
2,101441Mt Gibb JV ForrestaniaCCD001,10.0,5.011293,-0.275778,386.649325,93.000000,-59.799999
3,101441Mt Gibb JV ForrestaniaCCD001,15.0,7.515522,-0.404834,390.975080,92.900002,-60.000000
4,101441Mt Gibb JV ForrestaniaCCD001,20.0,10.012534,-0.526959,395.305208,92.699997,-60.000000
5,101441Mt Gibb JV ForrestaniaCCD001,25.0,12.513731,-0.640531,399.633153,92.500000,-59.900002
6,101441Mt Gibb JV ForrestaniaCCD001,30.0,15.011533,-0.745233,403.963278,92.300003,-60.099998
7,101441Mt Gibb JV ForrestaniaCCD001,35.0,17.498183,-0.845107,408.299935,92.300003,-60.200001
8,101441Mt Gibb JV ForrestaniaCCD001,40.0,19.980962,-0.946996,412.638763,92.400002,-60.200001
9,101441Mt Gibb JV ForrestaniaCCD001,45.0,22.463741,-1.048885,416.977590,92.300003,-60.200001


### Tangential

In [17]:
traces_tang = tangential_desurvey(collars_matched, surveys_matched, step=5.0)
print(f"Tangential: {len(traces_tang):,} trace points  |  {traces_tang[HOLE_ID].nunique()} holes")

# Compare end-point to minimum curvature for same hole
hole = traces_mc[HOLE_ID].iloc[0]
mc_end   = traces_mc[traces_mc[HOLE_ID]   == hole][["easting", "northing", "elevation"]].iloc[-1]
tang_end = traces_tang[traces_tang[HOLE_ID] == hole][["easting", "northing", "elevation"]].iloc[-1]
import numpy as np
drift_tang = np.sqrt(((mc_end - tang_end) ** 2).sum())
print(f"\nHole {hole} — displacement from minimum curvature at TD:")
print(f"  Tangential: {drift_tang:.2f} m")
traces_tang[traces_tang[HOLE_ID] == hole].head(5)

Tangential: 170,165 trace points  |  3687 holes

Hole 101441Mt Gibb JV ForrestaniaCCD001 — displacement from minimum curvature at TD:
  Tangential: 0.24 m


,hole_id,md,easting,northing,elevation,azimuth,dip
0,101441Mt Gibb JV ForrestaniaCCD001,0.0,0.000000,0.000000,378.000000,93.199997,-59.900002
1,101441Mt Gibb JV ForrestaniaCCD001,5.0,2.503644,-0.139975,382.325757,93.199997,-59.900002
2,101441Mt Gibb JV ForrestaniaCCD001,10.0,5.007287,-0.279951,386.651514,93.199997,-59.900002
3,101441Mt Gibb JV ForrestaniaCCD001,15.0,7.518940,-0.411581,390.972888,93.000000,-59.799999
4,101441Mt Gibb JV ForrestaniaCCD001,20.0,10.015739,-0.538063,395.303015,92.900002,-60.000000


### Balanced Tangential

In [18]:
traces_bt = balanced_tangential_desurvey(collars_matched, surveys_matched, step=5.0)
print(f"Balanced tangential: {len(traces_bt):,} trace points  |  {traces_bt[HOLE_ID].nunique()} holes")

# Spot-check end-point of one hole vs minimum curvature
hole = traces_mc[HOLE_ID].iloc[0]
mc_end  = traces_mc[traces_mc[HOLE_ID]  == hole][["easting", "northing", "elevation"]].iloc[-1]
bt_end  = traces_bt[traces_bt[HOLE_ID]  == hole][["easting", "northing", "elevation"]].iloc[-1]
import numpy as np
drift_bt = np.sqrt(((mc_end - bt_end) ** 2).sum())
print(f"\nHole {hole} — displacement from minimum curvature at TD:")
print(f"  Balanced tangential: {drift_bt:.2f} m")
traces_bt[traces_bt[HOLE_ID] == hole].head(5)

Balanced tangential: 170,165 trace points  |  3687 holes

Hole 101441Mt Gibb JV ForrestaniaCCD001 — displacement from minimum curvature at TD:
  Balanced tangential: 0.00 m


,hole_id,md,easting,northing,elevation,azimuth,dip
0,101441Mt Gibb JV ForrestaniaCCD001,0.0,0.000000,0.000000,378.000000,93.199997,-59.900002
1,101441Mt Gibb JV ForrestaniaCCD001,5.0,2.503644,-0.139975,382.325757,93.199997,-59.900002
2,101441Mt Gibb JV ForrestaniaCCD001,10.0,5.011296,-0.275785,386.649324,93.099998,-59.850000
3,101441Mt Gibb JV ForrestaniaCCD001,15.0,7.515527,-0.404835,390.975081,92.950001,-59.900000
4,101441Mt Gibb JV ForrestaniaCCD001,20.0,10.012543,-0.526959,395.305208,92.799999,-60.000000


## Attach 3D Positions

`attach_assay_positions` merges desurveyed trace coordinates onto assay intervals by nearest-depth lookup, giving each sample a 3D (E, N, Z) location.

In [19]:
# attach_assay_positions merges 3D trace coordinates onto each assay interval
# using a nearest-depth lookup (merge_asof on the interval midpoint)
ATTACH_HOLE = "97253ForrestaniaNMD140"   # datasource_hole_id = "NMD140"
assays_xyz = attach_assay_positions(
    assay_df[assay_df[HOLE_ID] == ATTACH_HOLE],
    traces_mc[traces_mc[HOLE_ID] == ATTACH_HOLE],
)
print(f"{ATTACH_HOLE}: {len(assays_xyz)} assay intervals with 3D coordinates attached")
assays_xyz[["hole_id", "from", "to", "mid", "au_ppm", "easting", "northing", "elevation"]].head()


97253ForrestaniaNMD140: 274 assay intervals with 3D coordinates attached


,hole_id,from,to,mid,au_ppm,easting,northing,elevation
0,97253ForrestaniaNMD140,160.02,160.52,160.270,0.007,-35.911442,1.627069,402.0
1,97253ForrestaniaNMD140,160.52,160.69,160.605,0.006,-35.911442,1.627069,402.0
2,97253ForrestaniaNMD140,160.69,160.90,160.795,0.002,-35.911442,1.627069,402.0
3,97253ForrestaniaNMD140,160.90,161.80,161.350,0.001,-35.911442,1.627069,402.0
4,97253ForrestaniaNMD140,161.80,162.30,162.050,0.004,-35.911442,1.627069,402.0


### Structural Positions

`attach_structure_positions` attaches 3D coordinates to structural point measurements using the same nearest-depth merge as the assay attachment.

In [20]:
# attach_structure_positions — same merge for structural point data
struct_subset = struct_df[struct_df[HOLE_ID] == struct_df[HOLE_ID].iloc[0]].copy()

# Build a dummy trace at constant azimuth/dip for demonstration
# (this hole is in the structure CSV but may not be in the survey CSV)
from baselode.drill.structural import attach_structure_positions

# Build a simple synthetic trace for this hole to demonstrate the function
import numpy as np
_hole = struct_subset[HOLE_ID].iloc[0]
_depths = np.arange(0, struct_subset["depth"].max() + 10, 5)
_synthetic_trace = pd.DataFrame({
    "hole_id":    _hole,
    "md":         _depths,
    "easting":    500000.0 + np.sin(np.radians(45)) * _depths,
    "northing":   6900000.0 + np.cos(np.radians(45)) * _depths,
    "elevation":  400.0 - np.sin(np.radians(60)) * _depths,
})

struct_xyz = attach_structure_positions(struct_subset, _synthetic_trace)
print(f"Structural positions attached: {len(struct_xyz)} rows")
struct_xyz[["hole_id", "depth", "dip", "azimuth", "easting", "northing", "elevation"]].head()

Structural positions attached: 169 rows


,hole_id,depth,dip,azimuth,easting,northing,elevation
0,101533ForrestaniaBD053,71.139999,NaN,NaN,500049.497475,6.900049e+06,339.378222
1,101533ForrestaniaBD053,71.809998,NaN,NaN,500049.497475,6.900049e+06,339.378222
2,101533ForrestaniaBD053,76.320000,NaN,NaN,500053.033009,6.900053e+06,335.048095
3,101533ForrestaniaBD053,76.559998,NaN,NaN,500053.033009,6.900053e+06,335.048095
4,101533ForrestaniaBD053,91.330002,NaN,NaN,500063.639610,6.900064e+06,322.057714


## Compositing

`composite_intervals` re-samples assay intervals onto fixed-length composites using length-weighted averaging. `resample_trace` does the same for 3D trace points.

In [21]:
# composite_intervals — re-samples assay intervals onto fixed-length composites
COMP_HOLE = "97253ForrestaniaNMD140"   # datasource_hole_id = "NMD140"
df_au = assay_df[assay_df[HOLE_ID] == COMP_HOLE][["hole_id", "from", "to", "au_ppm"]].dropna(subset=["au_ppm"])
print(f"{COMP_HOLE}: {len(df_au)} original Au intervals (irregular lengths)")
print(df_au.head())

composites_1m = composite_intervals(df_au, value_col="au_ppm", length=1.0, method="average")
print(f"\nAfter 1 m compositing: {len(composites_1m)} intervals")
print(composites_1m.head())

# resample_trace — re-sample a 3D trace at a different step
trace_nmd  = traces_mc[traces_mc[HOLE_ID] == COMP_HOLE]
resampled  = resample_trace(trace_nmd, step=10.0)
print(f"\nTrace {COMP_HOLE}: {len(trace_nmd)} points at 5 m step  →  {len(resampled)} points at 10 m step")
resampled.head()


97253ForrestaniaNMD140: 260 original Au intervals (irregular lengths)
                      hole_id    from      to  au_ppm
64356  97253ForrestaniaNMD140  160.02  160.52   0.007
64357  97253ForrestaniaNMD140  160.52  160.69   0.006
64358  97253ForrestaniaNMD140  160.69  160.90   0.002
64359  97253ForrestaniaNMD140  160.90  161.80   0.001
64360  97253ForrestaniaNMD140  161.80  162.30   0.004

After 1 m compositing: 253 intervals
                  hole_id    from      to   au_ppm
0  97253ForrestaniaNMD140  160.02  161.02  0.00506
1  97253ForrestaniaNMD140  161.02  162.02  0.00166
2  97253ForrestaniaNMD140  162.02  163.02  0.00688
3  97253ForrestaniaNMD140  163.02  164.02  0.00656
4  97253ForrestaniaNMD140  164.02  165.02  0.00744

Trace 97253ForrestaniaNMD140: 70 points at 5 m step  →  36 points at 10 m step


,hole_id,md,easting,northing,elevation
0,97253ForrestaniaNMD140,35.0,0.000000,0.000000,400.000000
1,97253ForrestaniaNMD140,45.0,-2.656998,0.120516,409.639789
2,97253ForrestaniaNMD140,55.0,-5.313995,0.241033,419.279578
3,97253ForrestaniaNMD140,65.0,-7.970993,0.361549,428.919367
4,97253ForrestaniaNMD140,75.0,-10.627991,0.482066,438.559156


## Structural Geometry Helpers

Convenience functions for normalising measurements and computing derived quantities.

In [22]:
from baselode.drill.structural import normalize_dip_azimuth, compute_strike, compute_plane_normal

# normalize_dip_azimuth — clamp dip to [0,90], azimuth to [0,360)
struct_norm = normalize_dip_azimuth(struct_df)
print("Before normalisation — dip min/max:", struct_df["dip"].min(), struct_df["dip"].max())
print("After  normalisation — dip min/max:", struct_norm["dip"].min(), struct_norm["dip"].max())
print()

# compute_strike — strike = (azimuth - 90) % 360
struct_norm["strike"] = compute_strike(struct_norm["azimuth"])
print("Strike sample:")
print(struct_norm[["hole_id", "depth", "dip", "azimuth", "strike"]].head())
print()

# compute_plane_normal — unit normal vector in ENU for a single measurement
dip_ex, az_ex = 45.0, 135.0
nx, ny, nz = compute_plane_normal(dip_ex, az_ex)
print(f"Plane normal for dip={dip_ex}° az={az_ex}°:  nx={nx:.4f}, ny={ny:.4f}, nz={nz:.4f}")

Before normalisation — dip min/max: 0.0 90.0
After  normalisation — dip min/max: 0.0 90.0

Strike sample:
                       hole_id      depth  dip  azimuth  strike
155960  101533ForrestaniaBD053  71.139999  NaN      NaN     NaN
155961  101533ForrestaniaBD053  71.809998  NaN      NaN     NaN
155962  101533ForrestaniaBD053  76.320000  NaN      NaN     NaN
155963  101533ForrestaniaBD053  76.559998  NaN      NaN     NaN
155964  101533ForrestaniaBD053  91.330002  NaN      NaN     NaN

Plane normal for dip=45.0° az=135.0°:  nx=0.5000, ny=-0.5000, nz=0.7071


## Interval algebra

`baselode.drill.intervals` is pure from-to algebra: lengths, midpoints, gap / overlap detection, split at boundary depths, clip to a window, and intersection-based table merge. Functions never mutate inputs.

In [23]:
import baselode.drill.intervals as intervals

gaps = intervals.detect_gaps(assay_df, min_gap=0.5)
overlaps = intervals.detect_overlaps(assay_df)
print(f'gaps:     {len(gaps):>5} rows')
print(f'overlaps: {len(overlaps):>5} rows')
gaps.head()

gaps:      2400 rows
overlaps:    95 rows


,hole_id,from,to,length
0,101533ForrestaniaBD048,148.00,158.00,10.00
1,101533ForrestaniaBD048,178.61,181.75,3.14
2,101533ForrestaniaBD048,190.80,196.36,5.56
3,101533ForrestaniaBD048,197.00,198.00,1.00
4,101533ForrestaniaBD048,205.02,210.95,5.93


In [24]:
hole_id = assay_df['hole_id'].iloc[0]
hole_intervals = assay_df[assay_df['hole_id'] == hole_id]
split = intervals.split_at(hole_intervals, depths={hole_id: [15.0, 35.0]})
print(f'before: {len(hole_intervals)} rows  after split @ 15 / 35 m: {len(split)} rows')

before: 55 rows  after split @ 15 / 35 m: 57 rows


## Comprehensive validation

`validate_drillhole_db(collar, survey, interval_tables)` runs every QC check in one pass and returns a structured `{summary, issues}` report — never raises. Each issue carries a `check`, severity, affected hole / table / row index, message, and a fix recipe pointing at the appropriate helper.

In [25]:
import baselode.drill.validate as validate

report = validate.validate_drillhole_db(
    collar_gdf,
    survey_df,
    interval_tables={'assay': assay_df},
)
print('summary:', report['summary'])

from collections import Counter
Counter(issue['check'] for issue in report['issues']).most_common()

summary: {'error': 2, 'warning': 2487, 'info': 2523}


[('interval_gaps', 2523),
 ('single_station_surveys', 2392),
 ('interval_overlaps', 95),
 ('azimuth_range', 2)]

In [26]:
survey_fixed = validate.fix_single_station_surveys(survey_df, collar_gdf)
survey_wrapped = validate.normalize_azimuth(survey_fixed)
print(f'before fix: {len(survey_df):>6} survey rows')
print(f'after  fix: {len(survey_fixed):>6} survey rows')

before fix: 158547 survey rows
after  fix: 160939 survey rows


## `DrillholeSet` — the composition root

Bundles collar + survey + N named interval tables into one object, mirroring PyGSLIB's `Drillhole(collar, survey)` + `addtable(...)` shape. Every method delegates to the existing function-based API; the trace is cached on the instance.

In [27]:
from baselode.drill import DrillholeSet

db = (
    DrillholeSet(collar_gdf, survey_df,
                 crs='EPSG:32750', project='gswa-tour')
      .add_table('assay', assay_df)
)
db

<DrillholeSet holes=4685 survey_rows=158547 tables=['assay']>

In [28]:
set_report = db.validate()
set_traces = db.desurvey(step=5.0)
print(f'validate.summary:    {set_report["summary"]}')
print(f'desurvey rows:       {len(set_traces)}')
print(f'cached on 2nd call:  {db.desurvey(step=5.0) is set_traces}')

validate.summary:    {'error': 2, 'warning': 2487, 'info': 2523}
desurvey rows:       170165
cached on 2nd call:  True


## OMF export

`baselode.drill.omf` round-trips drilling tables through Open Mining Format v1 (MIT-licensed, GMG-standard). Optional extra: `pip install baselode[omf]`. GSWA collars only carry lat/lon, so for OMF we build a projected collar table from the desurveyed trace heads.

In [29]:
from pathlib import Path

projected_collar = (
    set_traces.sort_values('md').groupby('hole_id').first().reset_index()
              [['hole_id', 'easting', 'northing', 'elevation']]
)
db_projected = DrillholeSet(projected_collar, survey_df).add_table('assay', assay_df)
db_projected._traces = set_traces

out_path = Path('/tmp/gswa-tour.omf')
db_projected.to_omf(out_path,
                    value_cols={'assay': [c for c in ['Cu_PPM', 'Au_PPM'] if c in assay_df.columns]})
f'{out_path}: {out_path.stat().st_size / 1024:.1f} KB'

'/tmp/gswa-tour.omf: 10216.8 KB'